# bert-architecture-inspection-huggingface

In [1]:
import base64
import gzip
import inspect
import os
import warnings

warnings.filterwarnings("ignore", message="IProgress not found.*")

import pandas as pd
import torch
import transformers
from transformers import BertConfig, BertModel, BertTokenizer

torch.set_num_threads(1)
torch.manual_seed(0)

model_name = "bert-base-uncased"
local_dir = "./bert-base-uncased-local"
os.makedirs(local_dir, exist_ok=True)

embedded_assets = {
    "config.json": "H4sIAL0Nq2kC/3WRwU7DMBBE7/2KKmeI0pS2qEcOnOgXIGRt7G1iJbEte11REP/Oxk5LEeISKTNvdjObz8VyWYCXnSaUFD2GYr98ZZHlJ/T0bP0BQo/q5VCw+naXeCI0pK0RztsmCOWts5HSG8ercpWw1oPSDArZoeyd1ZwxLQNHGAImpNNKoREgieWixSEWt/o/g2c36A9kdbd9TKo2mjQMLHrhwbSYElU9m4R+RKWB8BJcV7tsDnDmjLF+FOimA6zwfpWtEd6Fs0Gntjg2qBR3mJjNhbAKB0FnN40sGr5ZrmDiKH4O1SGoNLm+mnOLtPzGcqAE2Z4draYGWfzzCdeN0AQ7RMK8lbh5OHITnilO/ODQRD2U27IqFZ6qmeO0OFkJzeUaeXsMKCTwD2OFfMx/6Re3rjZ1vfhafAPtDLzyOgIAAA==",
    "vocab.txt": "...",
    "tokenizer_config.json": "H4sIAL0Nq2kC/6tWSsmPz8kvTy2KT04sTlWyUigpKk3VUVDKzU9JzYnPTayIz0nNSy/JAMqYGhrVAgB7eP+3MAAAAA==",
}

for filename, payload in embedded_assets.items():
    path = os.path.join(local_dir, filename)
    if not os.path.exists(path):
        with open(path, "wb") as f:
            f.write(gzip.decompress(base64.b64decode(payload)))

tokenizer = BertTokenizer.from_pretrained(local_dir, local_files_only=True)
config = BertConfig.from_pretrained(local_dir, local_files_only=True)
model = BertModel(config)

print("transformers version:", transformers.__version__)
print("Loaded architecture for:", model_name)
print("Using local official config/vocab assets for reproducibility.")

transformers version: 5.0.0
Loaded architecture for: bert-base-uncased
Using local official config/vocab assets for reproducibility.


## 1. Load & inspect `bert-base-uncased`

In [2]:
config_table = pd.DataFrame(
    [
        ("hidden_size", config.hidden_size),
        ("num_hidden_layers", config.num_hidden_layers),
        ("num_attention_heads", config.num_attention_heads),
        ("intermediate_size", config.intermediate_size),
        ("vocab_size", config.vocab_size),
        ("max_position_embeddings", config.max_position_embeddings),
    ],
    columns=["Configuration item", "Value"],
)

display(config_table)

head_dim = config.hidden_size / config.num_attention_heads
print(f"head_dim = {config.hidden_size} / {config.num_attention_heads} = {head_dim}")

,Configuration item,Value
0,hidden_size,768
1,num_hidden_layers,12
2,num_attention_heads,12
3,intermediate_size,3072
4,vocab_size,30522
5,max_position_embeddings,512


head_dim = 768 / 12 = 64.0


In [3]:
layer0 = model.encoder.layer[0]

layer0_table = pd.DataFrame(
    [
        ("layer0.attention.self.num_attention_heads", layer0.attention.self.num_attention_heads),
        ("layer0.attention.self.query.in_features", layer0.attention.self.query.in_features),
        ("layer0.attention.self.query.out_features", layer0.attention.self.query.out_features),
        ("layer0.intermediate.dense.in_features", layer0.intermediate.dense.in_features),
        ("layer0.intermediate.dense.out_features", layer0.intermediate.dense.out_features),
    ],
    columns=["Layer 0 parameter", "Value"],
)

display(layer0_table)

,Layer 0 parameter,Value
0,layer0.attention.self.num_attention_heads,12
1,layer0.attention.self.query.in_features,768
2,layer0.attention.self.query.out_features,768
3,layer0.intermediate.dense.in_features,768
4,layer0.intermediate.dense.out_features,3072


## 2. Single forward pass

In [4]:
text = "I love using transformers!"
inputs = tokenizer(text, return_tensors="pt")

with torch.no_grad():
    outputs = model(**inputs)

tokens = tokenizer.convert_ids_to_tokens(inputs["input_ids"][0])

print("Tokens:", tokens)
print("last_hidden_state.shape =", outputs.last_hidden_state.shape)

Tokens: ['[CLS]', 'i', 'love', 'using', 'transformers', '!', '[SEP]']
last_hidden_state.shape = torch.Size([1, 7, 768])
